In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('recent_dataset.csv')

In [3]:
# 조회수 분포의 하위 백분위수 확인
df['views'].describe(percentiles=[0.01, 0.03, 0.05, 0.5])

count     7292.000000
mean      1853.181706
std       4476.711414
min          9.000000
1%         108.000000
3%         148.000000
5%         180.000000
50%        752.000000
max      80594.000000
Name: views, dtype: float64

In [4]:
#스크랩 분포의 하위 백분위수 확인
df['scrap_count'].describe(percentiles=[0.01, 0.03, 0.05, 0.5])

count    7292.000000
mean       60.800192
std       155.482258
min         0.000000
1%          0.000000
3%          0.000000
5%          1.000000
50%        18.000000
max      2914.000000
Name: scrap_count, dtype: float64

- scrap_count/views일 때 views 하위 백분위수보다 scrap_count가 현저히 적기 때문에   임계값은 10으로 설정
- 기존에 만들어둔 파생피처 'scrap_rate' 컬럼에 필터 걸어봤을 때 10%넘는 수치가 없기 때문에 임계값 10 설정해서 기존 피처 수정


In [ ]:
#스크랩 비율 피처 생성 코드
df['scrap_rate (%)'] = np.where(
    df['views'] >= 10, 
    (df['scrap_count'] / df['views'] * 100).round(4), #views 수가 10 이상이면 정상 출력
    0 #views 수가 10 안넘으면 0 출력
)

# 접수기간 일수 피처 생성
# 문자열 날짜를 datetime 타입으로 변환
start_dt = pd.to_datetime(df['recruit_start'], format='%Y.%m.%d', errors='coerce')
end_dt = pd.to_datetime(df['recruit_end'], format='%Y.%m.%d', errors='coerce')

# 접수기간 일수 계산 (시작일과 종료일을 모두 포함하려면 + 1일)
df['recruit_period_days'] = (end_dt - start_dt).dt.days + 1

df.to_csv('recent_dataset.csv', index=False)

In [6]:
# 스크랩 비율 피처 잘 생성됐는지 확인하는 코드
# views가 10 미만인 샘플 (scrap_rate (%)가 전부 0이어야 함)
print("=== views < 10 샘플 ===")
display(df[df['views'] < 10][['views', 'scrap_count', 'scrap_rate (%)']].head())

# views가 10 이상인 샘플 (정상 계산된 값이 보여야 함)
print("\n=== views >= 10 샘플 ===")
display(df[df['views'] >= 10][['views', 'scrap_count', 'scrap_rate (%)']].head())

=== views < 10 샘플 ===


,views,scrap_count,scrap_rate (%)
7023,9,0,0.0



=== views >= 10 샘플 ===


,views,scrap_count,scrap_rate (%)
0,1217,50,4.1085
1,292,1,0.3425
2,349,0,0.0000
3,1009,40,3.9643
4,271,2,0.7380


'scrap_rate (%)' 피처 잘 생성됐는지 확인

In [7]:
print(df['recruit_period_days'].head())

0    63
1    39
2    52
3    39
4    24
Name: recruit_period_days, dtype: int64


'recruit_period_days'(접수기간 일수) 피처 잘 생성됐는지 확인